# Patent phrase matching: final offline submission

Utkrisht Suman · A24ARIU0001

Implementation prepared for review. Run the notebook to collect the experiment outputs; no training results or human ratings have been assumed.

# Patent phrase matching: starter notebook

This notebook is the **minimum working pipeline** for the mini project described in [project_brief.md](project_brief.md). It fine-tunes a pretrained Transformer **cross-encoder** to predict how similar two patent phrases are, checks it on a validation split, and writes `submission.csv`.

It is a starting point, not a solution. The baseline comparison, the experiments, and the error analysis in the brief are your work.

**Running it**

- Kaggle notebook, **GPU T4 or P100**, attached inputs:
  1. the competition data, via **Add Input → Competitions → U.S. Patent Phrase to Phrase Matching**;
  2. the output of [download_weights.ipynb](download_weights.ipynb), the saved pretrained checkpoint.
- Turn **Internet off** before you submit, because the competition runs submitted notebooks offline. The notebook falls back to the Hugging Face Hub only when it cannot find a local checkpoint, which works only with internet on.
- Writes to `/kaggle/working/`: `submission.csv` and `val_predictions.csv`. A rerun overwrites both. No checkpoints are saved.
- Uses packages preinstalled on Kaggle (`torch`, `transformers`, `datasets`, `scikit-learn`, `pandas`).

**Run package:** execute the full notebook. Written observations mentioning earlier CPU checks are reference observations; send your outputs back so they can be updated for your run.


In [ ]:
# Offline runtime bootstrap: weights and wheels must be attached as Kaggle inputs.
from pathlib import Path
import json, subprocess, sys
from importlib import metadata
if Path('/kaggle/input').exists():
    candidates=sorted(Path('/kaggle/input').rglob('patent_runtime_wheels/wheels_manifest.json'))
    assert candidates, 'Attach the complete saved output of PATENT_DOWNLOAD_WEIGHTS, including patent_runtime_wheels.'
    manifest_path=candidates[0];manifest=json.loads(manifest_path.read_text())
    expected_python=manifest['python'].split('.')[:2]
    assert expected_python==[str(sys.version_info.major),str(sys.version_info.minor)], 'Use the same Kaggle Python version as the weights/runtime preparation notebook.'
    missing=[]
    for spec in manifest['requirements']:
        name,version=spec.split('==',1)
        try: matches=metadata.version(name)==version
        except metadata.PackageNotFoundError: matches=False
        if not matches: missing.append(spec)
    if missing:
        loaded=any(name in sys.modules for name in ['numpy','pandas','transformers','datasets','sklearn'])
        subprocess.run([sys.executable,'-m','pip','install','-q','--no-index','--no-deps',
                        '--find-links',str(manifest_path.parent),*missing],check=True)
        if loaded:
            raise RuntimeError('Offline dependencies installed. Restart the kernel/session and Run All from the top once.')
    print('Offline runtime ready; no network or CUDA-wheel download used.')
else:
    print('Local patent run: install requirements.txt and set PATENT_* paths first.')


## 1. Setup

In [ ]:
import glob
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

import numpy as np
import pandas as pd
import torch
import transformers
from datasets import Dataset
from sklearn.model_selection import GroupShuffleSplit
from tokenizers.processors import TemplateProcessing
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, Trainer, TrainingArguments)

SEED = 42
transformers.set_seed(SEED)   # seeds Python, NumPy, and PyTorch; GPU results can still vary slightly between runs
device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| transformers", transformers.__version__, "| device:", device)
import inspect
# Preserve the starter 10% warmup across Transformers API versions.
WARMUP = ({"warmup_ratio": 0.1} if "warmup_ratio" in inspect.signature(TrainingArguments).parameters
          else {"warmup_steps": 0.1})

if Path("/kaggle/working").exists():
    assert torch.cuda.is_available(), "Enable a Kaggle T4 GPU before running the patent training notebook."


## 2. Configuration

These are the settings you will change in your experiments. Change **one** setting at a time so you can tell what caused a difference.

In [ ]:
import json, time, gc
from pathlib import Path
MODEL_ID = "microsoft/deberta-v3-small"
MAX_LEN, EPOCHS, LR, BATCH_SIZE, VAL_FRACTION = 64, 3, 2e-5, 32, 0.2
OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("patent_outputs")
OUT.mkdir(parents=True, exist_ok=True)
OUT_DIR = str(OUT)
def find_one(pattern):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    return os.path.dirname(hits[0]) if hits else None
DATA_DIR = os.environ.get("PATENT_DATA_DIR") or find_one("us-patent-phrase-to-phrase-matching/train.csv")
assert DATA_DIR, "Attach the competition data or set PATENT_DATA_DIR to its local folder."
MODEL_PATH = os.environ.get("PATENT_MODEL_PATH") or find_one("deberta-v3-small/config.json")
assert MODEL_PATH, "Attach offline pretrained weights or set PATENT_MODEL_PATH; no Hub fallback."
print("data:", DATA_DIR, "| offline model:", MODEL_PATH)

## 3. Data

Each row asks how similar a **target** phrase is to an **anchor** phrase, judged within a patent **context**. The context is a CPC (Cooperative Patent Classification) code such as `G02`, whose first letter is the section. Scores are 0, 0.25, 0.5, 0.75, or 1. The visible `test.csv` has only a few rows; when you submit, Kaggle reruns the notebook with the hidden test set in its place.

In [ ]:
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
print("train:", train.shape, "| test (visible):", test.shape)
print(train.score.value_counts().sort_index())
train.head()

## 4. Model input

The model reads the anchor as the first text segment, and the target plus a short context description as the second. The tokenizer joins them as `[CLS] anchor [SEP] target; context [SEP]`, so attention can compare the two phrases token by token: this is what makes it a cross-encoder. Here the context is just the CPC section name. How you represent context is one of the experiments you can run.

In [ ]:
config_path = os.environ.get("PATENT_CONFIG_PATH") or next(iter(sorted(
    glob.glob("/kaggle/input/**/best_config.json", recursive=True))), None)
assert config_path, "Attach the experiment output with best_config.json; do not choose by leaderboard score."
selection = json.loads(Path(config_path).read_text())
expected = {"seed": SEED, "epochs": EPOCHS, "lr": LR, "batch_size": BATCH_SIZE,
            "max_len": MAX_LEN, "val_fraction": VAL_FRACTION, "model_id": MODEL_ID}
assert all(selection.get(k) == v for k, v in expected.items()), "Final settings must match experiments"
assert selection.get("selected_by") == "validation Pearson"
CONTEXT_MODE = selection["context_mode"]
CPC_SECTIONS = {"A": "human necessities", "B": "performing operations; transporting",
    "C": "chemistry; metallurgy", "D": "textiles; paper", "E": "fixed constructions",
    "F": "mechanical engineering; lighting; heating; weapons; blasting", "G": "physics", "H": "electricity"}
def add_inputs(df):
    df = df.copy(); df["text_a"] = df.anchor
    if CONTEXT_MODE == "section": df["text_b"] = df.target + "; " + df.context.str[0].map(CPC_SECTIONS)
    elif CONTEXT_MODE == "none": df["text_b"] = df.target
    elif CONTEXT_MODE == "code": df["text_b"] = df.target + "; " + df.context
    else: raise ValueError(CONTEXT_MODE)
    return df
train, test = add_inputs(train), add_inputs(test)
print("Chosen from validation:", selection)

## 5. Validation split

The training set contains only about 730 distinct anchors, each paired with many targets. If the same anchor appeared in both the training and validation sets, the validation score would overestimate performance on unseen anchors. The split therefore holds out whole **anchors**.

Keep this split fixed (same `SEED` and `VAL_FRACTION`) for every experiment, so your comparisons are fair.

In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=VAL_FRACTION, random_state=SEED)
train_idx, val_idx = next(splitter.split(train, groups=train["anchor"]))
trn, val = train.iloc[train_idx].reset_index(drop=True), train.iloc[val_idx].reset_index(drop=True)
assert set(trn.anchor).isdisjoint(val.anchor)
print(f"train rows {len(trn)} | val rows {len(val)} | val anchors {val.anchor.nunique()}")

## 6. Tokenization

Padding is left to the data collator, which pads each batch only to its longest sequence. The printed example should start with `[CLS]` and have a `[SEP]` between the two segments.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
# transformers 5.0 builds this tokenizer without [CLS]/[SEP]; set the pair template explicitly
tokenizer.backend_tokenizer.post_processor = TemplateProcessing(
    single="[CLS] $A [SEP]", pair="[CLS] $A [SEP] $B:1 [SEP]:1",
    special_tokens=[("[CLS]", tokenizer.cls_token_id), ("[SEP]", tokenizer.sep_token_id)])


def to_dataset(df, with_labels=True):
    cols = ["text_a", "text_b"] + (["score"] if with_labels else [])
    ds = Dataset.from_pandas(df[cols].astype({"score": "float32"}) if with_labels else df[cols])  # MSE loss needs float32 labels
    ds = ds.map(lambda b: tokenizer(b["text_a"], b["text_b"], truncation=True, max_length=MAX_LEN),
                batched=True, remove_columns=["text_a", "text_b"])
    return ds.rename_column("score", "labels") if with_labels else ds


ds_trn, ds_val, ds_test = to_dataset(trn), to_dataset(val), to_dataset(test, with_labels=False)
print(tokenizer.decode(ds_trn[0]["input_ids"]))

## 7. Fine-tuning

With `num_labels=1`, the classification head outputs a single number. Because the labels are floats, Hugging Face treats the task as **regression** and minimizes mean squared error. The competition metric is the **Pearson correlation** between predictions and true scores. Pearson correlation ignores any linear rescaling of the predictions, so the model is rewarded for ranking pairs in the right order, not for hitting the exact values.

In [ ]:
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    return {"pearson": float(np.corrcoef(preds.reshape(-1), labels)[0, 1])}


# Load in fp32: the deberta-v3 checkpoints are stored in fp16, and fp16=True mixed precision below needs
# fp32 weights ("Attempting to unscale FP16 gradients" otherwise). Older transformers call this torch_dtype.
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH, num_labels=1, dtype=torch.float32, local_files_only=True)
args = TrainingArguments(
    output_dir=f"{OUT_DIR}/trainer_tmp",
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=2 * BATCH_SIZE,
    **WARMUP,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",          # keeps disk use and runtime low; the final weights stay in memory
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
)
trainer = Trainer(model=model, args=args, train_dataset=ds_trn, eval_dataset=ds_val,
                  data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=compute_metrics)
trainer.train()

## 8. Validation predictions

The predictions are saved so you can do your error analysis without retraining.

In [ ]:
val["pred"] = trainer.predict(ds_val).predictions.reshape(-1)
val_pearson = np.corrcoef(val["pred"], val["score"])[0, 1]
print(f"Validation Pearson: {val_pearson:.4f}")
val[["id", "anchor", "target", "context", "score", "pred"]].to_csv(f"{OUT_DIR}/val_predictions.csv", index=False)

## 9. Submission

The competition expects `/kaggle/working/submission.csv` with columns `id,score` and one row per test row. Clipping to [0, 1] keeps the scores in the label range; it does not change the validation Pearson correlation much.

In [ ]:
test_pred = trainer.predict(ds_test).predictions.reshape(-1)
submission = pd.DataFrame({"id": test["id"], "score": np.clip(test_pred, 0, 1)})
assert len(submission) == len(test) and submission["score"].notna().all()
submission.to_csv(f"{OUT_DIR}/submission.csv", index=False)
submission.head()

This final notebook reproduces the selected setting on the same grouped split. It intentionally retains the validation partition rather than silently changing the training budget after selection. It loads pretrained weights and configuration only from attached inputs; internet must be off. Save and run a version, submit its `submission.csv` to Kaggle, then record every public/private score and the version URL in the experiment ledger. No leaderboard score is assumed here.

## Download the result ZIP
The next cell collects small outputs only. Also download this notebook **with outputs** after the saved run finishes; Kaggle does not place its notebook file in /kaggle/working automatically.

In [ ]:
#!/usr/bin/env python3
"""Collect review outputs without datasets, environments or model weights. Standard library only."""
import argparse
from pathlib import Path
import zipfile
import json
import hashlib

SKIP_DIRS={'.venv','venv','.git','__pycache__','.cache','node_modules','site-packages',
           'patent_runtime_wheels','deberta-v3-small','all-MiniLM-L6-v2','data','adapter',
           'trainer','trainer_tmp','patent_weights','reference_evidence'}
JSON_NAMES={'best_config.json','run_summary.json','test_metrics.json','history.json','CHECKS.json'}
SKIP_FILES={'train.csv','test.csv','sample_submission.csv','IndianFoodDatasetCSV.csv',
            'tokenizer.json','config.json','tokenizer_config.json','adapter_config.json'}
SUFFIXES={'.ipynb','.csv','.png','.jpg','.jpeg','.pdf','.log','.txt'}

def eligible(path):
    if path.is_symlink() or not path.is_file():return False
    if any(p in SKIP_DIRS or p.startswith('checkpoint-') for p in path.parts):return False
    if path.name in SKIP_FILES:return False
    if path.suffix.lower()=='.json':return path.name in JSON_NAMES
    if path.suffix.lower() not in SUFFIXES:return False
    if path.suffix.lower()=='.txt' and path.name not in {'notebook_urls.txt','errors.txt','review_notes.txt'}:return False
    return True

def collect(roots,destination):
    destination=destination.resolve();destination.parent.mkdir(parents=True,exist_ok=True)
    count=0;manifest=[]
    with zipfile.ZipFile(destination,'w',zipfile.ZIP_DEFLATED) as archive:
        for index,root in enumerate(roots,1):
            root=root.resolve()
            if not root.exists():raise FileNotFoundError(root)
            paths=[root] if root.is_file() else sorted(root.rglob('*'))
            for path in paths:
                if path.resolve()==destination or not eligible(path):continue
                relative=Path(path.name) if root.is_file() else path.relative_to(root)
                arc=Path(f'{index:02d}_{root.stem}')/relative
                archive.write(path,str(arc));count+=1
                manifest.append({'path':str(arc),'bytes':path.stat().st_size,
                                 'sha256':hashlib.sha256(path.read_bytes()).hexdigest()})
        archive.writestr('RETURN_MANIFEST.json',json.dumps(manifest,indent=2))
    if count==0:
        destination.unlink();raise ValueError('No review outputs found. Check the input folder.')
    print(f'Saved {count} files: {destination} ({destination.stat().st_size/1e6:.2f} MB)',flush=True)


if Path("/kaggle/working").exists():
    collect([Path("/kaggle/working")],Path("/kaggle/working/A24ARIU0001_PATENT_FINAL_RESULTS.zip"))
